In [6]:
%pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 2.3 MB/s  0:00:002.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript] 1/2 [onnxscript]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: /home/lev/Projects/catbreed-helper/ai/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
import tensorflow as tf
import timm

/home/lev/Projects/catbreed-helper/ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import torch
import tensorflow as tf

device = torch.device('cpu')
state = torch.load('models/best_cat_breed_model.pth', map_location=device)
model = timm.create_model(
    'convnext_tiny',
    pretrained=False,           
    num_classes=66,
)
model.load_state_dict(state['model_state_dict'])

model.eval()

# TorchScript
example = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, example)
traced_model.save('models/model_traced.pt')

# PyTorch -> ONNX
torch.onnx.export(
    model, 
    example, 
    'models/model.onnx',
    input_names=['input'],     # явно указываем имя входа
    output_names=['output'],   # явно указываем имя выхода
    opset_version=11,
    dynamic_axes={
        'input': {0: 'batch'},
        'output': {0: 'batch'}
    }
)

W0525 00:12:05.692000 43388 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ConvNeXt([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ConvNeXt([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/lev/Projects/catbreed-helper/ai/.venv/lib/python3.13/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/lev/Projects/catbreed-helper/ai/.venv/lib/python3.13/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/lev/Projects/catbreed-helper/ai/.venv/lib/python3.13/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/lev/Projects/catbreed-helper/ai/.venv/lib/pyt

[torch.onnx] Optimize the ONNX graph...


Skipping constant folding for op SequenceEmpty with multiple outputs.


[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"linear_36"<FLOAT,[1,66]>
            ),
            initializers=(
                %"stem.0.bias"<FLOAT,[96]>{TorchTensor(...)},
                %"stem.1.weight"<FLOAT,[96]>{TorchTensor(...)},
                %"stem.1.bias"<FLOAT,[96]>{TorchTensor(...)},
                %"stages.0.blocks.0.conv_dw.bias"<FLOAT,[96]>{TorchTensor(...)},
                %"stages.0.blocks.0.norm.weight"<FLOAT,[96]>{TorchTensor(...)},
                %"stages.0.blocks.0.norm.bias"<FLOAT,[96]>{TorchTensor(...)},
                %"stages.0.blocks.0.mlp.fc1.bias"<FLOAT,[384]>{TorchTensor(...)},
                

In [8]:
import onnx

model = onnx.load('models/model.onnx')

print("Inputs:")
for inp in model.graph.input:
    print(f"  Name: '{inp.name}', Shape: {inp.type.tensor_type.shape}")

print("\nOutputs:")
for out in model.graph.output:
    print(f"  Name: '{out.name}', Shape: {out.type.tensor_type.shape}")

Inputs:
  Name: 'x', Shape: dim {
  dim_value: 1
}
dim {
  dim_value: 3
}
dim {
  dim_value: 224
}
dim {
  dim_value: 224
}


Outputs:
  Name: 'linear_36', Shape: dim {
  dim_value: 1
}
dim {
  dim_value: 66
}

